# [LAB] LangChain 실습

LangChain으로 체인을 만들고, 도구를 연결하고, 답을 정해진 모양으로 받고, 대화를 기억하게 합니다. 완성된 코드를 실행하고, 값을 바꿔 다시 실행하며 무엇이 달라지는지 봅니다.

- **2장 LCEL과 Runnable**: 프롬프트·모델·파서를 `|`로 잇고, 같은 체인을 여러 방식으로 실행합니다.
- **3장 도구 호출**: 모델이 도구를 고르고, 코드가 실행하고, 결과를 다시 모델에 넣는 반복을 따라가 봅니다.
- **4장 구조화된 출력**: 영화 리뷰를 정해진 모양으로 받는 두 방법을 비교하고, 받은 결과를 Python 값으로 써 봅니다.
- **5장 세션 메모리**: 세션별 대화 기록을 체인에 연결합니다.
- **6장 상태 안에 담는 메모리**: 같은 일을 LangGraph로 해 보고 5장과 비교합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. 실습 방법
- 2. LCEL과 Runnable
  - 2.1 기본 체인 만들기
  - 2.2 invoke·stream·batch로 실행하기
  - 2.3 RunnableParallel로 결과 모으기
  - 2.4 RunnablePassthrough로 값 덧붙이기
  - 2.5 RunnableLambda로 함수 끼우기
- 3. 도구 호출
  - 3.1 도구 정의 읽기
  - 3.2 도구 목록·연결·이름표 만들기
  - 3.3 도구 호출 루프 만들기
  - 3.4 질문해 보기
- 4. 구조화된 출력
  - 4.1 출력 스키마 읽기
  - 4.2 방법 1: PydanticOutputParser
  - 4.3 방법 2: with_structured_output
  - 4.4 결과를 Python 객체로 쓰기
- 5. 세션 메모리
  - 5.1 세션별 기록 저장소 만들기
  - 5.2 체인과 기록 연결하기
  - 5.3 세션 A·B로 대화해 보기
- 6. 상태 안에 담는 메모리
  - 6.1 State와 노드
  - 6.2 그래프 만들기
  - 6.3 스레드 A·B와 스냅샷
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

도구를 부를 때 `Error in ConsoleCallbackHandler.on_tool_start callback: KeyError('input')` 같은 줄이 섞여 보일 수 있습니다. 디버그 내용을 출력하는 쪽의 문제라 실행 결과에는 영향이 없으니 그대로 두면 됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/`의 5개 파일(6장만 솔루션). 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다. 노트북에서만 손본 곳(patch)은 장마다 메모에 적었습니다.
- LLM 호출이 많습니다(전체 약 25회, 2장만 약 10회). 사내 게이트웨이의 tool calling(3장)·json_schema 응답 형식(4.3) 지원은 미확인입니다.
- 값 바꾸기판(2026-10-18): 객관식을 빼고 완성 코드를 줍니다. CHANGE_HERE 11곳(2.1~2.5, 3.2, 3.4, 4.1, 4.3, 5.3, 6.3)의 "무슨 일이 생기나"는 사외 OpenAI(gpt-6-luna)로 실제 바꿔 실행한 결과입니다(모델 답이 달라지는 곳은 두 번씩). 사내 게이트웨이에서는 답 문구와 stream 조각 크기가 다를 수 있습니다.
- 4.4(결과를 Python 객체로 쓰기)는 원본에 없는 예제로, 사용자 요청(2026-10-18)으로 patch로 더했습니다.

</details>

## 1. 실습 방법

코드는 모두 완성되어 있습니다. 위에서부터 차례로 실행합니다.

- `# ✍️ CHANGE_HERE:` 주석이 있는 셀은 먼저 그대로 실행해 결과를 봅니다. 그다음 셀 위의 **바꿔 볼 값**을 복사해, 주석 바로 아래 줄의 표시된 자리(지금 값)를 지우고 붙여 넣습니다. 어떻게 달라질지 생각해 본 뒤 다시 실행합니다.
- 달라진 결과를 본 뒤 셀 아래 접힌 **무슨 일이 생기나**를 펼쳐 비교합니다. 확인했으면 값을 원래대로 되돌립니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. LCEL과 Runnable

프롬프트, 모델, 파서를 `|`로 이어 체인을 만들고, 같은 체인을 `invoke`·`stream`·`batch`로 실행합니다. 이어서 `RunnableParallel`·`RunnablePassthrough`·`RunnableLambda`로 결과를 모으고, 값을 덧붙이고, 함수를 체인에 끼워 봅니다.

2.2는 2.1에서 만든 `chain`을 실행하므로 2.1을 먼저 실행합니다.

**왜 배우나:** 체인은 뒤의 RAG와 Agent 코드에서도 계속 쓰는 기본 모양입니다. 직접 한 번 이어 보면 AI가 만든 긴 체인도 `|` 순서대로 읽을 수 있습니다.

준비 코드입니다. 체인에 쓸 도구(프롬프트 템플릿, 출력 파서, Runnable 조합 도구)를 불러오고 `model = get_llm()`으로 모델을 준비합니다. 접혀 있으니 펼치지 않아도 됩니다. 실행만 합니다.

In [ ]:
"""
LCEL Runnable: LangChain 구성 요소를 파이프라인으로 연결하기

핵심 개념:
  - Runnable = invoke, stream, batch를 공통으로 제공하는 실행 단위
  - LCEL = prompt → model → parser 흐름을 | 연산자로 선언하는 방식
  - RunnableParallel, RunnablePassthrough, RunnableLambda로 입력 흐름을 조합 가능

"""

import os

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)

from config.llm import get_llm


load_dotenv()

model = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/01_lcel_runnable.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 뺀 것: 이전 강사 말투 주석(`# 가우스`, `# 사용자가 입력한 값 -> …`, `# 사용자마다 서로 다른 내용의 주제를…`, `# 사용자 입력 -> 모델에게 전달 -> …`, `# 파이썬 주석처리 -> … ctrl + /`, `# 주의) 변수명이 pros, cons로 고정인게 아님! 이거 내가 맘음대로 지을 수 있음!`), 주석 처리된 `runnable.stream({'country': 'USA'})` 예시(조각을 키별로 이어 붙여 `pprint`), `translate_acronym` 안의 죽은 코드 2줄(`# response = model.invoke()`), docstring의 실행 경로.
- 솔루션을 따른 것: 주석 처리된 `invoke`/`stream`/`batch` 실행 줄을 실행되게 했고, 프롬프트 문구를 `너는 초보자를 위한 AI 튜터야.` / `{topic}을 초급자에게 3문장으로 설명해줘.`로 맞췄습니다(뜻 차이 없음).
- 뺀 주석 중 "키 이름은 마음대로 지을 수 있다"는 2장 볼 지점으로 옮겼습니다.
- CHANGE_HERE 5곳: 2.1 `3문장`→`1문장`(2.2 네 답이 한 문장), 2.2 `end=""`→`end=" | "`(stream 조각이 보임), 2.3 베트남→일본(pros·cons 둘 다), 2.4 질문→`LCEL이 뭐야?`(length 28→9), 2.5 GPT→LCEL/RAG(모델 없이 결정적).
- 2.2: 사내 게이트웨이가 stream을 조각으로 보내는지 미확인입니다. 한 번에 오면 `|`로 끊긴 모습이 거의 보이지 않습니다. 그때는 사외 결과("무슨 일이 생기나")를 보여 주며 설명합니다.
- 2.4 셀의 노드 이름 `Parallel<length>Input`은 `draw()`가 `&lt;`·`&gt;`로 바꿔 그려서, graphviz의 `Error: Unknown HTML element <length>` 줄이 나오지 않습니다(2026-10-19 고침).

</details>

### 2.1 기본 체인 만들기

**✍️ CHANGE_HERE: "3문장"을 "1문장"으로 바꾸면 2.2의 답이 어떻게 달라질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `3문장` 자리에 복사해 붙여 넣기

```text
1문장
```

먼저 2.2까지 그대로 실행해 본 뒤, 값을 바꿔 이 셀과 2.2 셀을 차례로 다시 실행합니다. 확인했으면 `3문장`으로 되돌리고 이 셀을 다시 실행합니다.

`ChatPromptTemplate.from_messages([...])` 안에 메시지 두 개를 `(역할, 내용)` 튜플로 넣습니다. system 메시지에는 "초보자를 위한 AI 튜터"라는 역할을, human 메시지에는 `{topic}` 자리가 있는 질문을 담습니다. 중괄호 `{topic}`은 실행할 때 값이 들어갈 빈자리입니다.

`|`로 프롬프트 → 모델 → 파서를 차례로 잇습니다. 적은 순서가 곧 데이터가 흐르는 방향입니다.

In [ ]:
# ── 기본 LCEL 체인 ────────────────────────────────────────────────────────────
# ChatPromptTemplate: `{}` 변수만 바꿔 재사용할 수 있는 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 초보자를 위한 AI 튜터야."),
        # ✍️ CHANGE_HERE: "3문장"을 "1문장"으로 바꾸면 2.2의 답이 어떻게 달라질까요?
        ("human", "{topic}을 초급자에게 3문장으로 설명해줘."),
    ]
)

# StrOutputParser: LLM 출력을 문자열로 변환
parser = StrOutputParser()

# LCEL: LangChain Expression Language
# - 프롬프트 작성 → LLM 호출 → 출력 파싱 과정을 하나의 실행 단위로 묶는다.
# - prompt, model, parser는 모두 Runnable 계열 객체다.
# - Runnable | Runnable 결과도 Runnable이므로 체인을 계속 이어붙일 수 있다.
chain = prompt | model | parser

<details><summary>무슨 일이 생기나</summary>

2.2의 네 답이 모두 한 문장이 됩니다(바꾸기 전에는 세 문장 안팎, 두 번 실행해 두 번 모두). 예: `LLM은 방대한 글을 학습해 사람처럼 문장을 이해하고 이어서 쓰도록 만들어진 인공지능입니다.`

`invoke`·`stream`·`batch`가 모두 같은 `chain`, 같은 프롬프트 템플릿을 쓰기 때문입니다. 템플릿 한 곳을 고치면 그 체인을 쓰는 모든 실행에 반영됩니다.

</details>

### 2.2 invoke·stream·batch로 실행하기

**✍️ CHANGE_HERE: end=""를 end=" | "로 바꾸면 stream 답이 어떻게 보일까요?**

**바꿔 볼 값** — 바로 아래 줄의 `end=""` 자리에 복사해 붙여 넣기

```text
end=" | "
```

이 셀을 다시 실행하고 두 `----` 줄 사이의 stream 답을 봅니다. 끝나면 `end=""`로 되돌려 둡니다(2.4 이후에 이 셀을 다시 실행하려면 2.1부터 실행합니다).

In [ ]:
# Runnable 실행 방식:
# - invoke(): 단일 입력 처리
# - stream(): 토큰/청크 단위 출력
# - batch(): 여러 입력 일괄 처리

answer = chain.invoke({"topic": "LLM"})
print(answer)
print("----")
for chunk in chain.stream({"topic": "LangChain"}):
    # ✍️ CHANGE_HERE: end=""를 end=" | "로 바꾸면 stream 답이 어떻게 보일까요?
    print(chunk, end="", flush=True)
print()

print("----")
inputs = [{"topic": "LangChain Expression Language"}, {"topic": "LangChain Runnable"}]
responses = chain.batch(inputs)
for topic_input, response in zip(inputs, responses):
    print(f"{topic_input.get('topic')}: {response.strip()}")

<details><summary>무슨 일이 생기나</summary>

가운데 `stream` 답만 `Lang | Chain | 은 |  AI |  모델 | 을 | …`처럼 조각마다 `|`로 끊겨 출력됩니다(두 번 실행해 두 번 모두). 모델이 답을 한 번에 보내지 않고 몇 글자씩 나눠 보낸다는 것이 보입니다. 맨 앞과 맨 끝에는 빈 조각이 있어 `|`가 겹쳐 보이기도 합니다.

`invoke`와 `batch`의 답은 그대로 이어진 글입니다. 두 방식은 답을 다 받은 뒤 한 번에 돌려줍니다. 글자가 조금씩 나타나는 챗봇 화면은 `stream` 방식으로 만듭니다.

</details>

### 2.3 RunnableParallel로 결과 모으기

**✍️ CHANGE_HERE: "베트남"을 "일본"으로 바꾸면 pros와 cons 중 어느 쪽이 바뀔까요?**

**바꿔 볼 값** — 바로 아래 줄의 `베트남` 자리에 복사해 붙여 넣기

```text
일본
```

모델을 네 번 호출해 출력이 깁니다. 이 셀을 다시 실행해 위쪽 `pros`·`cons` 목록의 첫 줄만 보고, 다 봤으면 `베트남`으로 되돌립니다.

In [ ]:
# ── RunnableParallel ──────────────────────────────────────────────────────────
# 여러 Runnable을 병렬로 실행해 결과를 dict로 모을 때 사용
pros_chain = (
    ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 장점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser()
)
cons_chain = (
    ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 단점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser()
)

runnable = RunnableParallel(pros=pros_chain, cons=cons_chain)
# ✍️ CHANGE_HERE: "베트남"을 "일본"으로 바꾸면 pros와 cons 중 어느 쪽이 바뀔까요?
answers = runnable.invoke({"country": "베트남"})
print("pros")
print(answers.get("pros"))

print("cons")
print(answers.get("cons"))


# Python dict가 pipe 사이에 들어가면 자동으로 RunnableParallel처럼 동작한다.
parallel_dict = RunnablePassthrough() | {
    "pros": ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 장점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser(),
    "cons": ChatPromptTemplate.from_template(
        "{country}로 여행갈 때의 단점을, 5개 정도 목록 형식으로 설명해줘."
    )
    | model
    | StrOutputParser(),
}

answers = parallel_dict.invoke({"country": "베트남"})
print("pros")
print(answers.get("pros"))

print("cons")
print(answers.get("cons"))

<details><summary>무슨 일이 생기나</summary>

위쪽 `pros`와 `cons`가 **둘 다** 일본 이야기로 바뀝니다. 예: pros `가까운 거리와 짧은 비행시간`, cons `비용이 많이 들 수 있음`. `RunnableParallel`이 입력 `{"country": "일본"}` 하나를 두 체인에 똑같이 넣기 때문입니다.

아래쪽 dict 방식은 `"베트남"` 그대로라 베트남 장단점이 나옵니다. 목록 문구는 실행할 때마다 조금씩 다릅니다.

</details>

### 2.4 RunnablePassthrough로 값 덧붙이기

**✍️ CHANGE_HERE: 질문을 "LCEL이 뭐야?"로 바꾸면 length가 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `RunnablePassthrough의 역할이 뭐야?` 자리에 복사해 붙여 넣기

```text
LCEL이 뭐야?
```

이 셀을 다시 실행해 첫 줄의 dict만 봅니다. 본 뒤에는 원래 질문으로 되돌립니다.

In [ ]:
# ── RunnablePassthrough ───────────────────────────────────────────────────────
# 입력값을 그대로 다음 단계에 전달하면서 부가 필드를 추가할 때 사용
def get_question_length(text: str):
    return len(text)


# 예: 사용자 질문을 바탕으로 맥락 정보나 메타데이터를 추가
passthrough = RunnablePassthrough.assign(
    length=lambda x: get_question_length(x["question"])
)
# ✍️ CHANGE_HERE: 질문을 "LCEL이 뭐야?"로 바꾸면 length가 어떻게 될까요?
print(passthrough.invoke({"question": "RunnablePassthrough의 역할이 뭐야?"}))

prompt = ChatPromptTemplate.from_template("{question} 3문장으로 설명해줘.")
chain = passthrough | prompt | model | StrOutputParser()

from draw import draw

draw(chain)  # 체인 구조를 그림으로 봅니다
print(chain.invoke({"question": "RunnablePassthrough의 역할이 뭐야?"}))

<details><summary>무슨 일이 생기나</summary>

첫 줄이 `{'question': 'LCEL이 뭐야?', 'length': 9}`가 됩니다(바꾸기 전에는 `length`가 28). 입력 `question`은 그대로 지나가고 `length`만 새로 계산해 덧붙습니다. 띄어쓰기와 물음표도 한 글자로 셉니다.

아래쪽 체인에 넣는 질문은 바꾸지 않았으므로 마지막 답은 그대로 RunnablePassthrough 설명입니다.

</details>

### 2.5 RunnableLambda로 함수 끼우기

**✍️ CHANGE_HERE: "GPT"를 "LCEL"이나 사전에 없는 "RAG"로 바꾸면 무엇이 나올까요?**

**바꿔 볼 값** — 바로 아래 줄의 `GPT` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

사전에 있는 약어
```text
LCEL
```

사전에 없는 약어
```text
RAG
```

값마다 이 셀을 다시 실행해 첫 줄을 봅니다. 끝나면 `GPT`로 되돌립니다.

In [ ]:
# ── RunnableLambda ────────────────────────────────────────────────────────────
# 일반 Python 함수를 Runnable로 감싸 체인 안에서 사용할 때 활용
acronym_dict = {
    "GPT": "Generative Pre-trained Transformer",
    "LLM": "Large Language Model",
    "LCEL": "LangChain Expression Language",
}


def translate_acronym(acronym: str):
    return acronym_dict.get(acronym, acronym)


# ✍️ CHANGE_HERE: "GPT"를 "LCEL"이나 사전에 없는 "RAG"로 바꾸면 무엇이 나올까요?
print(RunnableLambda(translate_acronym).invoke("GPT"))

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 초보자를 위한 AI 튜터야."),
        ("human", "{topic}을 초급자에게 3문장으로 설명해줘."),
    ]
)

chain = (
    {
        "topic": RunnableLambda(translate_acronym),
    }
    | prompt
    | model
    | StrOutputParser()
)
print(chain.invoke("GPT"))

<details><summary>무슨 일이 생기나</summary>

`"LCEL"`은 첫 줄이 `LangChain Expression Language`로, 사전에 없는 `"RAG"`는 `RAG` 그대로 나옵니다. 사전에 없으면 받은 글자를 그대로 돌려주는 보통 Python 함수라, 몇 번 실행해도 결과가 같습니다.

아래쪽 체인에는 여전히 `"GPT"`를 넣으므로 마지막 답은 GPT 설명 그대로입니다. `RunnableLambda`로 감싸면 이 함수가 체인 안에서도 한 단계로 똑같이 동작합니다.

</details>

**볼 지점**
- 2.2: 같은 `chain`을 호출 방법만 바꿔 실행했습니다. `invoke`는 답을 한 번에, `stream`은 조각으로 이어서, `batch`는 두 주제의 답을 목록으로 돌려줍니다.
- 2.3: `RunnableParallel`은 두 체인의 결과를 `pros`·`cons` 키가 있는 dict 하나로 모읍니다. dict를 `|` 사이에 두어도 같은 일을 합니다. 키 이름 `pros`, `cons`는 정해진 이름이 아니라 마음대로 지을 수 있습니다.
- 2.4: `RunnablePassthrough.assign`은 입력을 그대로 넘기면서 `length` 값을 덧붙입니다.
- 2.5: `RunnableLambda`로 감싼 함수는 체인 안에서 한 단계가 됩니다. 마지막 체인은 약어를 풀어 쓴 이름을 `{topic}` 자리에 넣습니다.

<details><summary>강사용 메모</summary>

- 2.3 dict 방식의 `RunnablePassthrough() | {...}`는 dict 앞에 Runnable 하나를 두어 dict가 RunnableParallel로 바뀌게 한 모양입니다.
- 셀을 건너뛰면 2.4·2.5에서 `prompt`·`chain` 이름이 다시 정의되므로, 2.2를 다시 실행하려면 2.1부터 다시 실행합니다.

</details>

## 3. 도구 호출

모델에게 날씨 조회 도구와 계산 도구를 알려 주고, 질문에 맞는 도구를 모델이 고르게 합니다. 모델이 고른 도구는 코드가 실행하고, 결과를 다시 모델에게 넣어 최종 답을 받습니다.

3.1은 도구를 만드는 코드입니다. `@tool`을 붙인 함수의 docstring(함수 첫 줄의 설명 글)은 모델에게 주는 도구 설명이 되고, 타입 힌트(`city: str`)는 인자 타입이 됩니다. 3.2에서 도구 목록을 모델에 연결하고, 3.3의 반복이 모델이 고른 도구를 실행하고, 3.4에서 질문을 보냅니다.

**왜 배우나:** Agent가 도구를 쓰는 방식이 바로 이 반복입니다. 등록 → 선택 → 실행 → 결과 전달을 한 번 직접 따라가 보면, 도구가 늘어나도 같은 구조로 읽을 수 있습니다.

준비 코드입니다. 메시지 종류와 도구를 만드는 `@tool`을 불러옵니다. 펼치지 않고 실행만 해도 됩니다.

In [ ]:
"""
Tool Calling: LLM이 외부 도구를 자율적으로 선택·실행하기

핵심 흐름:
  사용자 질문 → LLM이 필요한 도구 결정 → 도구 실행 → 결과 반환

"""

import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

from config.llm import get_llm

load_dotenv();

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/02_tool_calling.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 뺀 것: 이전 강사 주석(description은 LLM을 위한 설명, 타입 힌트도 LLM에게 힌트, "그건 어떻게하는데?"), `bind_tools` 주석 2줄("시퀀스를 전달받을 수 있음", "한번에 여러 도구들"), `tool_map` 위의 `'''{'get_weahter': …}'''` 문자열(오타 포함), 경로 주석 `05-langgraph/03_tool_node.py 참고`(2026-09-28 사용자 결정), docstring의 실행 경로. "docstring·타입 힌트가 모델에게 주는 힌트"라는 내용은 장 소개 글로 옮겼습니다.
- `tool_map`은 솔루션 표기 `{t.name: t for t in tools}`를 씁니다(`tool` 이름이 데코레이터 `@tool`과 겹치지 않게).
- `print(call)`은 raw에만 있는 줄이지만 호출 정보(name·args·id)를 화면에 보여 주려고 남겼습니다. 솔루션에는 없습니다.
- CHANGE_HERE 2곳: 3.2 `tools`에서 `calculate`를 뺌(계산 질문의 호출 줄이 사라지고 모델이 직접 계산, 사외 2/2 정답 1,158), 3.4 서울→제주/대전(대전은 도구 결과 "찾을 수 없습니다"로 모른다고 답함, 사외 2/2). 사내 모델은 대전 날씨를 지어낼 수도 있습니다(미확인).

</details>

### 3.1 도구 정의 읽기

In [ ]:
# ── 도구 정의 (@tool 데코레이터) ────────────────────────────────────────────────
# 함수의 docstring이 LLM에게 전달되는 도구 설명이 된다.
# 타입 힌트가 JSON Schema로 변환되어 파라미터 형식을 LLM에게 알려준다.


@tool
def get_weather(city: str) -> str:
    """지정한 도시의 현재 날씨를 조회한다."""
    fake_data = {
        "서울": "맑음, 기온 22°C",
        "부산": "흐림, 기온 24°C",
        "제주": "비, 기온 20°C",
    }
    return fake_data.get(city, f"{city}의 날씨 데이터를 찾을 수 없습니다.")


@tool
def calculate(expression: str) -> str:
    """수학 수식을 계산한다. 예: '2 + 3 * 4'"""
    try:
        result = eval(expression, {"__builtins__": {}})  # noqa: S307
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"

### 3.2 도구 목록·연결·이름표 만들기

**✍️ CHANGE_HERE: 목록에서 calculate를 빼면 3.4의 계산 질문은 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄에서 `, calculate` 지우기

이 셀과 3.4 셀을 차례로 다시 실행합니다. 확인했으면 `, calculate`를 다시 넣고 이 셀을 다시 실행합니다.

3.1에서 `@tool`로 만든 두 도구를 목록 하나에 담습니다. 이 목록이 모델에게 알려 줄 도구 명단입니다.

`bind_tools()`는 모델을 호출할 때마다 도구 명세(이름·설명·인자)를 함께 보내게 합니다. 결과는 새 이름 `llm_with_tools`에 담습니다. 3.3의 루프가 이 이름을 씁니다.

모델은 쓸 도구를 이름(문자열)으로만 알려 줍니다. 그래서 이름을 넣으면 실제 도구가 나오는 dict(`{이름: 도구}`)를 미리 만들어 둡니다.

In [ ]:
# ✍️ CHANGE_HERE: 목록에서 calculate를 빼면 3.4의 계산 질문은 어떻게 될까요?
tools = [get_weather, calculate]

# ── LLM에 도구 바인딩 ─────────────────────────────────────────────────────────
# bind_tools()는 LLM 호출 시마다 도구 스키마를 자동으로 포함시킨다.
llm = get_llm()
llm_with_tools = llm.bind_tools(tools)

# 도구 이름 → 실행 함수 매핑
tool_map = {t.name: t for t in tools}

<details><summary>무슨 일이 생기나</summary>

3.4의 계산 질문에서 `{'name': 'calculate', …}` 호출 정보 줄이 사라지고, 모델이 `(123 + 456) × 2 = 579 × 2 = **1,158**`처럼 혼자 계산해 답합니다(두 번 실행해 두 번 모두). 날씨 질문은 그대로 `get_weather`를 호출합니다.

모델은 `bind_tools()`로 알려 준 도구 안에서만 고릅니다. 이번 답은 맞았지만 도구로 계산한 것이 아니라서 틀릴 수도 있습니다. 답만 보지 말고 호출 정보 줄로 도구를 썼는지 확인합니다.

</details>

### 3.3 도구 호출 루프 만들기

`response.tool_calls`의 항목 `call` 하나에는 도구 이름(`name`), 인자(`args`), 호출 번호(`id`)가 들어 있습니다. 이름으로 `tool_map`에서 도구를 찾고, 인자를 넣어 실행한 결과를 `result`에 담습니다. 바로 아래 줄이 `result`를 `ToolMessage`로 만들어 대화에 되돌려 넣습니다.

In [ ]:
# ── 수동 도구 호출 루프 ───────────────────────────────────────────────────────
# LLM 응답에 tool_calls가 있으면 실행 후 결과를 다시 LLM에 전달한다.
# LangGraph의 ToolNode가 이 루프를 자동화한다.


def run_with_tools(question: str) -> str:
    messages = [HumanMessage(content=question)]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            print(call)
            tool_fn = tool_map[call["name"]]
            result = tool_fn.invoke(call["args"])
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"],
                )
            )

### 3.4 질문해 보기

**✍️ CHANGE_HERE: "서울"을 "제주"나 데이터에 없는 "대전"으로 바꾸면 답이 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `서울` 자리에 복사해 붙여 넣기 (하나씩 바꿔 실행)

가짜 날씨 데이터에 있는 도시
```text
제주
```

가짜 날씨 데이터에 없는 도시
```text
대전
```

값마다 이 셀을 다시 실행해 첫 번째 질문의 결과를 봅니다. 다 봤으면 `서울`로 되돌려 둡니다.

In [ ]:
# ── 실행 예시 ─────────────────────────────────────────────────────────────────

print("=== 날씨 질문 ===")
# ✍️ CHANGE_HERE: "서울"을 "제주"나 데이터에 없는 "대전"으로 바꾸면 답이 어떻게 될까요?
print(run_with_tools("서울 날씨 알려줘"))

print("\n=== 수학 계산 ===")
print(run_with_tools("(123 + 456) * 2를 계산해줘"))

print("\n=== 복합 질문 ===")
print(run_with_tools("서울과 부산 날씨를 비교해줘"))

# LangGraph ToolNode 방식으로 전환하면 이 루프가 필요 없어진다.

<details><summary>무슨 일이 생기나</summary>

`"제주"`: 호출 정보 줄의 인자가 `{'city': '제주'}`로 바뀌고, 답이 도구 결과대로 `제주는 현재 비가 내리고 있고, 기온은 20°C예요.`가 됩니다.

`"대전"`: 모델은 똑같이 `get_weather`를 `{'city': '대전'}`으로 호출합니다. 도구가 `대전의 날씨 데이터를 찾을 수 없습니다.`를 돌려주자 모델은 `현재 대전 날씨 정보를 조회하지 못했어요.`처럼 모른다고 답했습니다(두 번 실행해 두 번 모두). 최종 답은 도구 결과(`ToolMessage`)를 바탕으로 만들어집니다.

</details>

**볼 지점**
- 호출 정보 줄의 `name`은 모델이 고른 도구, `args`는 모델이 채운 인자, `id`는 호출 번호입니다. 모델은 도구를 실행하지 않고 이 정보만 돌려줍니다. 도구 이름과 인자를 `tool`·`arguments`라고 적는 자료도 있습니다. 이름만 다르고 담긴 정보는 같습니다.
- 도구를 실행한 것은 `run_with_tools()` 안의 코드입니다. 실행 결과는 `ToolMessage`로 대화에 들어가고, 모델이 그것을 읽고 최종 답을 만듭니다.
- 복합 질문(서울과 부산)에서는 호출 정보 줄이 두 번 나올 수 있습니다. 모델이 도구를 두 번 고른 것입니다.

<details><summary>강사용 메모</summary>

- 사내 게이트웨이에서 tool calling(`bind_tools`)이 동작하는지 미확인입니다. 안 되면 3.2·3.4의 "무슨 일이 생기나"만 읽고 넘어갑니다.
- 슬라이드 JSON(`tool`/`arguments`)과 호출 정보(`name`/`args`/`id`)의 이름이 다릅니다. 슬라이드를 가리키며 대응을 짚어 줍니다.
- `calculate`는 `eval`을 씁니다(builtins 비움). 수업용 가짜 도구입니다. 물으면 "실서비스에서는 쓰지 않는다"고만 답합니다.
- LangGraph의 `ToolNode`가 이 루프를 자동화합니다(2일차 `src/raw/Chapter01/04-langgraph/03_tool_node.py`).

</details>

## 4. 구조화된 출력

영화 리뷰를 글이 아니라 `MovieReview`라는 정해진 모양(제목·장르·평점·요약·추천 여부)으로 받습니다. 두 가지 방법을 씁니다.

- 방법 1 `PydanticOutputParser`(4.2): 프롬프트에 형식 지시를 넣고, 모델의 글 답을 파서가 읽어 `MovieReview`로 바꿉니다.
- 방법 2 `with_structured_output()`(4.3): 모델에게 스키마를 직접 넘겨 형식 지시 없이 `MovieReview`를 받습니다.

4.1의 스키마에서 필드마다 붙은 `description`이 모델에게 전달되는 설명입니다. 4.4에서는 받은 결과를 Python 값으로 바로 써 봅니다.

**왜 배우나:** 답이 글로 오면 사람이 읽어야 하지만, 정해진 모양으로 오면 프로그램이 바로 쓰는 데이터가 됩니다. 평점으로 계산하거나 거르고, dict나 JSON 글자로 바꿔 저장하거나 다른 프로그램에 넘길 수 있습니다.

준비 코드입니다. 파서, 프롬프트 템플릿, 스키마를 만드는 Pydantic 도구를 불러오고 모델을 준비합니다. 내용은 몰라도 되니 실행하고 넘어갑니다.

In [ ]:
"""
Output Parser: LLM 출력을 구조화된 데이터로 변환하기

두 가지 방식 비교:
  1. PydanticOutputParser + format_instructions → 프롬프트에 형식 지시 포함
  2. .with_structured_output()                 → LLM의 function-calling 기능 활용 (권장)

"""

from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/03_output_parser.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 방법 2는 raw에서 전부 주석(TODO 2곳 포함)이라 솔루션 코드로 채웠습니다. PDF에 없는 개념이라 예제로 둡니다(2026-09-28 사용자 결정).
- 솔루션을 따른 것: system 프롬프트를 `…응답해.\n{format_instructions}`로(형식 지시가 줄을 바꿔 들어감), human 프롬프트를 `{movie} 리뷰 작성해줘`로.
- 뺀 것: `# review1은 MovieReview의 인스턴스가 되길 희망한다.` 주석, 끝의 ASCII 비교 표(장 끝 "볼 지점"의 표로 옮김), docstring의 실행 경로.
- CHANGE_HERE 2곳: 4.1 평점 설명 `1.0 ~ 10.0`→`1.0 ~ 5.0`(4.2·4.3 평점 9.0→4.5·4.8, 사외 2/2. `ge`/`le`가 없어 검사는 안 함), 4.3 기생충→`반도체 2099`(없는 영화도 다섯 칸을 지어냄, 사외 2/2). 사내 모델이 칸에 "정보 없음"처럼 쓰면, 그것도 모양은 지키고 내용은 모델에 달렸다는 예로 씁니다.
- 4.4는 원본에 없는 예제입니다(사용자 요청, 2026-10-18, patch로 4.3 뒤에 더함). `review1`(4.2)과 `review2`(4.3)를 쓰므로 4.2·4.3을 먼저 실행해야 합니다. LLM을 부르지 않습니다.

</details>

### 4.1 출력 스키마 읽기

**✍️ CHANGE_HERE: 평점 설명을 "1.0 ~ 5.0"으로 바꾸면 4.2·4.3의 평점이 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `10.0` 자리에 복사해 붙여 넣기

```text
5.0
```

이 셀과 4.2, 4.3 셀을 차례로 다시 실행합니다. 확인했으면 `10.0`으로 되돌리고 이 셀부터 4.3까지 다시 실행합니다.

In [ ]:
# ── 출력 스키마 정의 ───────────────────────────────────────────────────────────


class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    genre: str = Field(description="장르 (예: 액션, 드라마, SF)")
    # ✍️ CHANGE_HERE: 평점 설명을 "1.0 ~ 5.0"으로 바꾸면 4.2·4.3의 평점이 어떻게 될까요?
    rating: float = Field(description="평점 (1.0 ~ 10.0)")
    summary: str = Field(description="100자 이내 한 줄 요약")
    recommend: bool = Field(description="추천 여부")

<details><summary>무슨 일이 생기나</summary>

4.2와 4.3의 평점이 모두 5 이하로 나옵니다. 바꾸기 전 `9.0`이던 평점이 `4.5`, `4.8`처럼 바뀌었습니다(두 번 실행해 두 번 모두).

`description`의 글은 4.2에서는 형식 지시 안에, 4.3에서는 모델에게 넘기는 스키마 안에 들어가 모델에게 전달됩니다. 필드 설명이 곧 모델에게 주는 지시입니다. 다만 범위를 글로 알려 줬을 뿐 코드가 검사하지는 않으므로, 범위를 꼭 지켜야 하면 받은 값을 코드로 한 번 더 확인합니다.

</details>

### 4.2 방법 1: PydanticOutputParser

`PydanticOutputParser`에 4.1의 `MovieReview`를 알려 주면, 모델의 글 답을 `MovieReview`로 바꾸는 파서가 됩니다.

system 메시지에 `{format_instructions}` 자리를 두고, `.partial()`로 그 자리에 파서가 만든 형식 지시를 미리 채워 둡니다. human 메시지에는 `{movie}` 자리가 있습니다.

2장에서 만든 체인과 같은 모양입니다. 이번에는 마지막 단계가 문자열 파서가 아니라 `MovieReview` 파서입니다.

In [ ]:
# ── 방법 1: PydanticOutputParser ──────────────────────────────────────────────
# 프롬프트에 JSON 형식 지시문을 주입하여 LLM이 구조화된 출력을 생성하도록 유도.
# LLM이 지시를 따르지 않으면 파싱 실패 가능.

print("=== 방법 1: PydanticOutputParser ===")

parser = PydanticOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "영화 리뷰 전문가야. 다음 형식으로 응답해.\n{format_instructions}"),
        ("human", "{movie} 리뷰 작성해줘"),
    ]
).partial(format_instructions=parser.get_format_instructions())

chain1 = prompt | llm | parser
review1: MovieReview = chain1.invoke({"movie": "인터스텔라"})

print(f"제목   : {review1.title}")
print(f"장르   : {review1.genre}")
print(f"평점   : {review1.rating}")
print(f"요약   : {review1.summary}")
print(f"추천   : {'✓' if review1.recommend else '✗'}")

### 4.3 방법 2: with_structured_output

**✍️ CHANGE_HERE: "기생충"을 세상에 없는 영화 이름으로 바꾸면 다섯 칸이 어떻게 채워질까요?**

**바꿔 볼 값** — 바로 아래 줄의 `기생충` 자리에 복사해 붙여 넣기

```text
반도체 2099
```

이 셀을 다시 실행합니다. 본 뒤에는 `기생충`으로 되돌려 둡니다.

In [ ]:
# ── 방법 2: .with_structured_output() ────────────────────────────────────────
# LLM의 function-calling(tool use) 기능을 활용하므로 더 안정적.
# 프롬프트에 형식 지시를 추가할 필요 없음.

print("\n=== 방법 2: with_structured_output() ===")

structured_llm = llm.with_structured_output(MovieReview)

prompt2 = ChatPromptTemplate.from_messages(
    [
        ("system", "영화 리뷰 전문가야."),
        ("human", "{movie} 리뷰 작성해줘"),
    ]
)

chain2 = prompt2 | structured_llm
# ✍️ CHANGE_HERE: "기생충"을 세상에 없는 영화 이름으로 바꾸면 다섯 칸이 어떻게 채워질까요?
review2: MovieReview = chain2.invoke({"movie": "기생충"})

print(f"제목   : {review2.title}")
print(f"장르   : {review2.genre}")
print(f"평점   : {review2.rating}")
print(f"요약   : {review2.summary}")
print(f"추천   : {'✓' if review2.recommend else '✗'}")

<details><summary>무슨 일이 생기나</summary>

세상에 없는 영화인데도 다섯 칸이 모두 채워집니다. 모델이 줄거리를 지어내지는 않고 요약 칸에 `제목만으로는 작품 정보를 확인하기 어려워 …`라고 적습니다(여섯 번 실행해 여섯 번 모두). 그래도 장르 `SF`, 평점 `5.0`(`6.5`가 나오기도 함), 추천 `✗`는 제목만 보고 짐작해 넣은 값입니다.

구조화된 출력은 답의 **모양**(다섯 칸과 칸마다의 타입)을 지켜 줄 뿐, **내용**이 사실인지는 보장하지 않습니다. 모르는 영화여도 칸을 비워 두지 않고 값을 채우므로, 칸이 다 찼다고 맞는 답은 아닙니다. 내용은 따로 검토합니다.

</details>

### 4.4 결과를 Python 객체로 쓰기

In [ ]:
# ── 결과를 Python 객체로 쓰기 ─────────────────────────────────────────────────
import json

# review2는 글이 아니라 MovieReview 객체입니다
print(type(review2))

# 1) 점(.) 뒤에 필드 이름을 써서 값을 꺼냅니다
print(review2.title, review2.rating)

# 2) rating은 숫자라서 바로 계산합니다: 4.2에서 받은 review1과의 평점 차이
print("review1과 평점 차이:", round(review2.rating - review1.rating, 1))

# 3) recommend는 참/거짓(True/False)이라 if에 바로 씁니다
if review2.recommend:
    print("추천 목록에 넣습니다:", review2.title)
else:
    print("추천 목록에서 뺍니다:", review2.title)

# 4) dict로 바꿉니다: 필드 이름이 키가 됩니다
review_dict = review2.model_dump()
print(review_dict)

# 5) JSON 글자로 바꿉니다: 파일에 저장하거나 다른 프로그램에 보낼 때 씁니다
# ensure_ascii=False는 한글을 \uXXXX 같은 기호로 바꾸지 않고 그대로 둡니다
review_json = json.dumps(review_dict, ensure_ascii=False)
print(review_json)

**볼 지점**
- 두 방법 모두 결과가 `MovieReview`라서 `review1.title`처럼 필드 이름으로 값을 꺼냅니다.
- 방법 1은 프롬프트의 형식 지시에 기대고, 방법 2는 프롬프트에 형식 지시가 없습니다.
- 4.4: `review2`는 `MovieReview` 객체라서 `review2.rating`처럼 값을 꺼내 바로 계산하고 `if`에 씁니다. `model_dump()`는 dict로, `json.dumps(..., ensure_ascii=False)`는 JSON 글자로 바꿉니다. 모델의 답이 프로그램이 바로 쓰는 데이터가 된 것입니다.

두 방법을 비교하면 다음과 같습니다.

| | PydanticOutputParser | with_structured_output() |
|---|---|---|
| 안정성 | LLM 지시 이행에 의존 | function-calling 보장 |
| 프롬프트 복잡도 | format_instructions 필요 | 프롬프트 단순 |
| 구형 모델 지원 | ✓ (모든 텍스트 LLM) | function-calling 필요 |
| 권장 | 구형 모델 / 커스텀 파싱 | GPT-4o 이상 기본 권장 |

<details><summary>강사용 메모</summary>

- 비교 표는 원본 끝의 ASCII 표(`print('''┌───…''')`)를 그대로 옮겼습니다. ASCII 그림은 셀에서 뺐습니다(2026-09-28 사용자 결정).
- 표와 원본 주석의 "function-calling 보장", "GPT-4o 이상 기본 권장"은 원본 문구입니다. 현재 langchain-openai(1.6.6)의 `with_structured_output` 기본 방식은 `json_schema`(응답 형식 지정)이지 function calling이 아닙니다. 물으면 "모델이 스키마에 맞춰 답하도록 API 차원에서 지정한다"고 답합니다.
- 사내 게이트웨이의 json_schema 응답 형식 지원은 미확인입니다. 방법 2가 실패하면 `llm.with_structured_output(MovieReview, method="function_calling")`으로 바꿔 볼 수 있습니다(노트북 코드는 원본대로 둠).

</details>

## 5. 세션 메모리

대화 기록을 체인 밖의 저장소에 세션(대화방)별로 모아 두고, 요청할 때마다 그 기록을 프롬프트에 넣어 줍니다. 대화를 그대로 보관해 다시 넣는 이 방식을 Buffer Memory라고 부릅니다.

5.1에서 세션별 기록 저장소를 만들고, 5.2에서 체인과 연결한 뒤, 5.3에서 두 세션으로 대화합니다.

**왜 배우나:** 모델은 앞 대화를 기억하지 않습니다. 이 장의 `RunnableWithMessageHistory`가 "이전 대화를 다시 넣어 보내기"를 대신 해 줍니다.

준비 코드입니다. 대화 기록을 담는 도구와 `RunnableWithMessageHistory`를 불러오고 모델을 준비합니다. 접혀 있으니 실행만 합니다.

In [ ]:
"""
메모리 전략 1 — LangChain 방식: RunnableWithMessageHistory

대화 기록을 체인 외부(InMemoryChatMessageHistory)에서 세션별로 관리.
체인 자체는 순수 함수처럼 동작하고, 히스토리는 wrapper가 주입한다.

"""

from dotenv import load_dotenv
from langchain_core.chat_history import (
    BaseChatMessageHistory,
    InMemoryChatMessageHistory,
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core._api import LangChainDeprecationWarning
import warnings

# 수업 화면 정리: 이 장에서 쓰는 두 도구의 사용 중단 예정 안내 경고만 숨깁니다.
warnings.filterwarnings(
    "ignore",
    message="RunnableWithMessageHistory is deprecated",
    category=LangChainDeprecationWarning,
)
warnings.filterwarnings(
    "ignore",
    message="The class `InMemoryChatMessageHistory` was deprecated",
    category=LangChainDeprecationWarning,
)

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/02-langchain/04_memory_runnable.py`. 원본 파일은 그대로 두고 노트북에서만 손봤습니다(patch).
- 뺀 것: `get_session_history` 안의 강사 말투 주석 4줄, 대화 누적 모양을 적은 `'''# 최초의 ChatPromptTemplate …'''` 문자열(5.2 설명의 text 블록으로 옮김), docstring의 실행 경로.
- 마지막 출력의 `LangGraph 방식(05_memory_langgraph.py)`을 `LangGraph 방식`으로 줄였습니다(2026-09-28 사용자 결정).
- 접힌 준비 코드에 경고 숨김 줄을 더했습니다. langchain-core 1.6.4에서 `RunnableWithMessageHistory`(5.2, "RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.")와 `InMemoryChatMessageHistory`(5.3 첫 호출, "The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 …")가 사용 중단 예정 경고를 냅니다. 사용자 결정(2026-09-28)으로 이 두 경고만(`LangChainDeprecationWarning` + 메시지 일치) 숨겼고 다른 경고는 그대로 보입니다. 결정 당시에는 앞의 것만 알려졌고, 뒤의 것은 실행 검증에서 발견해 같은 이유로 함께 숨겼습니다. 물으면 "LangChain이 LangGraph 방식을 권장한다는 안내"라고 답하고 6장으로 잇습니다.
- CHANGE_HERE: 5.3 `config_b`의 `session_B`→`session_A`(세션 B가 홍길동을 앎, 저장 메시지 4→6줄, 사외 2/2). 옛 CHANGE_HERE(첫 질문의 이름 바꾸기)는 차이가 말투뿐이라 옮겼습니다. 5.1부터 다시 실행해야 기록이 비워집니다.

</details>

### 5.1 세션별 기록 저장소 만들기

`get_session_history(session_id)`는 세션 번호를 받아 그 세션의 대화 기록을 돌려주는 함수입니다. 처음 보는 세션이면 빈 기록(`InMemoryChatMessageHistory()`)을 새로 만들어 `session_store`에 넣고, 이미 있으면 있던 기록을 돌려줍니다.

In [ ]:
# ── 세션 저장소 ───────────────────────────────────────────────────────────────
# 실제 서비스에서는 Redis·DB로 교체 가능

session_store: dict[str, BaseChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

### 5.2 체인과 기록 연결하기

`RunnableWithMessageHistory`는 체인을 감싸서, 호출 전에 세션 기록을 프롬프트의 `history` 자리에 넣고 호출 뒤에 새 대화를 기록에 더합니다. 사용자 입력이 든 키 이름(`input`)과 기록을 넣을 자리 이름(`history`)을 알려 줘야 합니다.

같은 세션에서 두 번 물으면 요청이 이렇게 쌓입니다.

```text
첫 번째 요청: system, human(첫 번째 질문)
첫 번째 응답: ai(첫 번째 응답) → 기록에 저장
두 번째 요청: system, human(첫 번째 질문), ai(첫 번째 응답), human(두 번째 질문)
```

In [ ]:
# ── 체인 구성 ─────────────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "너는 친절한 AI 어시스턴트야."),
        MessagesPlaceholder(variable_name="history"),  # 이전 대화가 여기 삽입됨
        ("human", "{input}"),
    ]
)

chain = prompt | llm

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

### 5.3 세션 A·B로 대화해 보기

**✍️ CHANGE_HERE: "session_B"를 "session_A"로 바꾸면 세션 B의 답이 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `session_B` 자리에 복사해 붙여 넣기

```text
session_A
```

5.1 셀부터 이 셀까지 차례로 다시 실행합니다(5.1이 저장된 기록을 비웁니다). 확인했으면 `session_B`로 되돌리고 5.1부터 다시 실행합니다.

In [ ]:
# ── 세션 A ────────────────────────────────────────────────────────────────────
print("=== 세션 A: 홍길동 ===")
config_a = {"configurable": {"session_id": "session_A"}}

r1 = chain_with_history.invoke({"input": "내 이름은 홍길동이야."}, config=config_a)
print(f"AI: {r1.content}")

r2 = chain_with_history.invoke({"input": "내 이름이 뭐야?"}, config=config_a)
print(f"AI: {r2.content}")


# ── 세션 B ────────────────────────────────────────────────────────────────────
print("\n=== 세션 B: 별개 대화 ===")
# ✍️ CHANGE_HERE: "session_B"를 "session_A"로 바꾸면 세션 B의 답이 어떻게 될까요?
config_b = {"configurable": {"session_id": "session_B"}}

r3 = chain_with_history.invoke({"input": "내 이름이 뭐야?"}, config=config_b)
print(f"AI: {r3.content}")  # B는 A 대화를 모름


# ── 저장된 히스토리 확인 ──────────────────────────────────────────────────────
print("\n=== 세션 A 저장된 메시지 ===")
for msg in get_session_history("session_A").messages:
    role = "사람" if msg.type == "human" else " AI "
    print(f"[{role}] {msg.content}")


# ── LangChain 방식의 한계 ─────────────────────────────────────────────────────
print("""
[LangChain 방식 한계]
- 히스토리가 체인 외부에 있어 그래프와 통합이 어려움
- 도구 실행 결과가 메시지 흐름에 자동 포함되지 않음
- 복잡한 분기·반복 구조에서 히스토리 관리가 번거로움

→ LangGraph 방식은 State 안에 messages를 두어 이 문제를 해결.
""")

<details><summary>무슨 일이 생기나</summary>

세션 B의 답이 `홍길동님이에요!`처럼 이름을 압니다(두 번 실행해 두 번 모두). 바꾸기 전에는 `아직 이름을 알려주지 않으셔서 몰라요.`였습니다. "세션 A 저장된 메시지"도 4줄에서 6줄로 늘어 세 번째 질문과 답이 함께 쌓입니다.

모델은 그대로이고 바뀐 것은 `session_id` 하나입니다. `RunnableWithMessageHistory`는 `session_id`로 `session_store`에서 기록을 골라 프롬프트에 넣으므로, 같은 이름을 쓰면 같은 대화가 됩니다.

</details>

**볼 지점**
- 세션 A는 같은 `session_id`로 두 번 물었기 때문에 두 번째 요청에 첫 대화가 함께 들어갔습니다.
- 세션 B는 `session_id`가 달라 기록이 비어 있습니다. 모델이 같아도 기록이 다르면 다른 대화가 됩니다.
- 기록은 체인 밖의 `session_store`에 있습니다. 마지막 출력은 이 방식의 한계를 적고, 다음 장의 LangGraph 방식으로 넘어갑니다.

## 6. 상태 안에 담는 메모리

같은 일을 LangGraph로 합니다. 대화 기록을 그래프의 State(그래프가 들고 다니는 정보 묶음) 안 `messages`에 두고, `MemorySaver`(스레드별 State를 저장해 두는 장치)가 스레드(대화방)별로 저장합니다. 5장과 이 장은 모두 대화를 그대로 쌓아 다시 넣는 Buffer 방식입니다. 다른 점은 기록을 두는 곳입니다.

LangGraph는 2일차에 자세히 배우므로 여기서는 완성된 코드를 실행하고 5장과 비교합니다. 6.1의 `chatbot` 노드는 system 메시지와 State의 `messages`를 함께 모델에 보내고 답을 `messages`에 더합니다. 6.2는 그 노드 하나를 시작(START)과 끝(END) 사이에 두고 `MemorySaver`와 함께 그래프를 만듭니다.

**왜 배우나:** Agent를 만들 때는 대화 기록뿐 아니라 도구 결과, 분기 같은 흐름도 함께 관리해야 합니다. State 안에 기록을 두면 이것들을 한곳에서 다룰 수 있습니다.

준비 코드입니다. LangGraph 도구와 메시지 종류를 불러오고 모델을 준비합니다. 이 셀도 실행을 잊지 않습니다.

In [ ]:
"""
메모리 전략 2 — LangGraph 방식: State에 messages 내장

대화 기록이 State의 일부 → MemorySaver가 스레드별로 영속화.
외부 히스토리 관리 불필요. 도구 결과도 자동으로 메시지 흐름에 포함.

"""

from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본: `src/solution/Chapter01/02-langchain/05_memory_langgraph.py`(raw는 TODO 3곳이 빈 상태: `return {'messages': []}`, `checkpointer = None`, `app = None`). LangGraph는 2-1에서 배우므로 예제로 실행만 합니다(목차 결정). 노트북에서만 손본 것: 끝의 ASCII 비교 표를 빼고 장 끝 "볼 지점"의 표로 옮김, docstring의 실행 경로 삭제.
- 같은 커널에서 6.3을 다시 실행하면 스레드 A 기록이 계속 쌓입니다. 처음 상태로 보려면 6.2부터 다시 실행합니다(`MemorySaver`를 새로 만듦).
- 원본은 마지막 답만 `print(...content)`로, 스냅샷은 `[사람]`·`[ AI ]` 한 줄씩 출력합니다. 노트북은 결과 변수를 `memory_graph_result_a`, `memory_graph_result_b`로 바꾸고, 메시지 목록을 `results`로 꺼내 `pretty_print()`로 출력합니다(2026-10-17 규칙, dev/AGENTS.md).
- CHANGE_HERE: 6.3 `thread_B`→`thread_A`(스레드 B 메시지 2→6칸, 스냅샷 4→6칸, 사외 2/2). 6.2부터 다시 실행해야 저장된 State가 비워집니다.

</details>

### 6.1 State와 노드

In [ ]:
# ── State 정의 ────────────────────────────────────────────────────────────────
# add_messages reducer: 기존 메시지 리스트에 새 메시지를 누적(append) 한다.
# 덮어쓰기 방지 → 대화 히스토리가 자동으로 쌓인다.


class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: State) -> dict:
    system = SystemMessage(content="너는 친절한 AI 어시스턴트야.")
    return {"messages": [llm.invoke([system] + state["messages"])]}

### 6.2 그래프 만들기

In [ ]:
# ── 그래프 빌드 ───────────────────────────────────────────────────────────────
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

# MemorySaver: 스레드별 State를 메모리에 유지 (프로세스 재시작 시 초기화)
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

### 6.3 스레드 A·B와 스냅샷

**✍️ CHANGE_HERE: "thread_B"를 "thread_A"로 바꾸면 스레드 B의 답과 메시지 칸 수가 어떻게 될까요?**

**바꿔 볼 값** — 바로 아래 줄의 `thread_B` 자리에 복사해 붙여 넣기

```text
thread_A
```

6.2 셀과 이 셀을 차례로 다시 실행합니다(6.2가 저장된 State를 비웁니다). 확인했으면 `thread_B`로 되돌리고 6.2부터 다시 실행합니다.

In [ ]:
# ── 스레드 A: 홍길동 ───────────────────────────────────────────────────────────
print("=== 스레드 A ===")
config_a = {"configurable": {"thread_id": "thread_A"}}

app.invoke(
    {"messages": [HumanMessage(content="내 이름은 홍길동이야.")]}, config=config_a
)
memory_graph_result_a = app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")]}, config=config_a
)
results = memory_graph_result_a["messages"]
for m in results:
    m.pretty_print()


# ── 스레드 B: 별개 세션 ───────────────────────────────────────────────────────
print("\n=== 스레드 B (A와 독립) ===")
# ✍️ CHANGE_HERE: "thread_B"를 "thread_A"로 바꾸면 스레드 B의 답과 메시지 칸 수가 어떻게 될까요?
config_b = {"configurable": {"thread_id": "thread_B"}}

# B는 A 대화를 모름
memory_graph_result_b = app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")]}, config=config_b
)
results = memory_graph_result_b["messages"]
for m in results:
    m.pretty_print()


# ── 스냅샷 확인 ───────────────────────────────────────────────────────────────
print("\n=== 스레드 A 현재 State 스냅샷 ===")
snapshot = app.get_state(config_a)
results = snapshot.values["messages"]
for m in results:
    m.pretty_print()

print(f"\n다음 실행 노드: {snapshot.next}")  # () = 완료 상태

<details><summary>무슨 일이 생기나</summary>

"스레드 B" 아래 메시지 칸이 2칸에서 6칸으로 늘어 스레드 A의 앞 대화(`내 이름은 홍길동이야.` …)가 함께 나오고, 마지막 답도 `홍길동님이에요.`가 됩니다(두 번 실행해 두 번 모두). 스냅샷도 4칸에서 6칸이 됩니다.

같은 `thread_id`로 호출하면 `MemorySaver`가 저장해 둔 State를 이어 쓰기 때문입니다. 5.3과 달리 그래프 결과의 `messages`에 대화 전체가 들어 있어 쌓인 기록이 한눈에 보입니다.

</details>

**볼 지점**
- `add_messages`는 새 메시지를 덮어쓰지 않고 `messages` 뒤에 붙입니다. 그래서 대화가 State 안에 쌓입니다.
- 같은 `thread_id`로 호출하면 `MemorySaver`가 저장해 둔 State를 이어 씁니다. 그래서 스레드 A는 이름을 기억합니다.
- `app.get_state(config_a)`로 스레드 A의 State를 꺼내 저장된 대화를 볼 수 있습니다. `다음 실행 노드: ()`는 그래프 실행이 끝났다는 뜻입니다.

5장 방식과 비교하면 다음과 같습니다.

| | LangChain (RunnableWithMessageHistory) | LangGraph (MemorySaver) |
|---|---|---|
| 히스토리 위치 | 체인 외부 (session_store) | State 내부 (messages 필드) |
| 도구 결과 통합 | 수동 처리 필요 | 자동 (ToolMessage로 축적) |
| 분기/반복 | 구현 복잡 | 그래프 구조로 자연스럽게 표현 |
| 영속성 | 별도 저장소 연결 필요 | checkpointer 교체만으로 전환 |
| 권장 시나리오 | 단순 Q&A 챗봇 | Agent, 복잡한 대화 흐름 |

<details><summary>강사용 메모</summary>

- 비교 표는 원본 끝의 ASCII 표를 그대로 옮겼습니다(ASCII 그림은 셀에서 뺌, 2026-09-28 사용자 결정). 원본 표의 첫 열 이름 `RunnableWithHist`는 줄임말이라 전체 이름으로 적었습니다.

</details>

## 7. 정리

체인, 도구 호출, 구조화된 출력, 메모리를 이어 보았습니다.

| 장 | 무엇을 연결했나 | 결과가 어디에 나오나 |
| --- | --- | --- |
| 2. LCEL과 Runnable | 프롬프트·모델·파서를 파이프 연산자로, 여러 체인을 dict로 | 답변 텍스트, pros·cons 두 목록 |
| 3. 도구 호출 | 도구 목록을 모델에(`bind_tools`), 도구 이름을 실제 도구에(`tool_map`) | 호출 정보 줄과 도구 결과를 쓴 최종 답 |
| 4. 구조화된 출력 | 스키마를 파서에, 또는 모델에 직접 | 제목·장르·평점·요약·추천 다섯 줄, dict와 JSON 글자 |
| 5. 세션 메모리 | 체인과 세션 기록 저장소를 | 이름을 기억하는 세션 A, 모르는 세션 B |
| 6. 상태 안에 담는 메모리 | 그래프와 `MemorySaver`를 | 스레드별 대화와 State 스냅샷 |

모델은 도구를 실행하지 않고, 이전 대화를 기억하지도 않습니다. 실행과 기억은 코드가 맡고, LangChain과 LangGraph는 그 일을 정해진 모양으로 묶어 줍니다.

값을 바꿔 다시 실행하고, 달라진 결과를 "무슨 일이 생기나"와 비교했습니다. 값 하나를 바꿔 보면 그 줄이 결과의 어디를 정하는지 보입니다.